# LLM Prompting & Insight Generation

This notebook uses retrieved customer feedback evidence to generate
grounded, structured business insights using a large language model.

In [1]:
import pandas as pd
import numpy as np

## Load Retrieved Data & Embeddings

We load the chunked customer feedback and precomputed embeddings
to support retrieval-augmented prompting.

In [2]:
chunked_df = pd.read_csv("../data/processed/review_chunks.csv")
embeddings = np.load("../data/processed/review_embeddings.npy")

## LLM Strategy

We use the LLM strictly as a reasoning and summarization layer.
All factual grounding comes from retrieved customer feedback.

In [ ]:
#from openai import OpenAI
#client = OpenAI()

OpenAIError: The api_key client option must be set either by passing api_key to the client or by setting the OPENAI_API_KEY environment variable

## Prompt Template

The prompt enforces:
- Evidence-based reasoning
- Structured output
- Business-friendly language

In [5]:
def mock_llm(prompt):
    return (
        "Summary of Customer Issues:\n"
        "- Pricing and billing concerns appear frequently\n"
        "- Customers mention lack of clarity in charges\n\n"
        "Potential Business Impact:\n"
        "- Increased churn risk among price-sensitive customers\n"
        "- Reduced trust and satisfaction\n\n"
        "Recommended Actions:\n"
        "- Improve billing transparency\n"
        "- Proactively communicate pricing changes"
    )

## Retrieve Evidence for Business Query

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer

# Load embedding model
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

def retrieve_evidence(query, top_k=5):
    query_embedding = embedding_model.encode([query])
    similarities = cosine_similarity(query_embedding, embeddings)[0]
    top_indices = similarities.argsort()[-top_k:][::-1]
    return chunked_df.iloc[top_indices]["text_chunk"].tolist()

/usr/local/python/3.12.1/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
def build_prompt(query, evidence_chunks):
    context = "\n".join([f"- {chunk}" for chunk in evidence_chunks])

    prompt = f"""
You are a customer analytics expert.

Answer the following question using ONLY the provided customer feedback evidence.

Question:
{query}

Customer Feedback Evidence:
{context}

Your response should include:
1. Key reasons
2. Business impact
3. Actionable recommendations
"""

    return prompt.strip()

In [11]:
query = "Why are customers at risk of churn?"

evidence = retrieve_evidence(query)
prompt = build_prompt(query, evidence)

# Option A: OpenAI
# response = client.chat.completions.create(
#     model="gpt-4o-mini",
#     messages=[{"role": "user", "content": prompt}]
# )
# print(response.choices[0].message.content)

# Option B: Mock
print(mock_llm(prompt))


Summary of Customer Issues:
- Pricing and billing concerns appear frequently
- Customers mention lack of clarity in charges

Potential Business Impact:
- Increased churn risk among price-sensitive customers
- Reduced trust and satisfaction

Recommended Actions:
- Improve billing transparency
- Proactively communicate pricing changes


## Output Interpretation

- The LLM summarizes issues grounded in retrieved evidence.
- Outputs are explainable and auditable.
- This approach avoids hallucinations and unsupported claims.

## Business Use Cases

- Automated churn risk explanations
- Customer feedback summarization for leadership
- Decision support for retention and product teams